<a href="https://colab.research.google.com/github/Madhu-1807/ExplainableAI_SHAP_LIME/blob/main/XAI_SHAP_LIME_Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import pickle

**Step 1: Loading Dataset**

In [ ]:
# Load data
df = pd.read_excel("/content/drive/MyDrive/ISPY_DataPaper_features.xlsx")

# Drop rows where PCR is missing
df = df.dropna(subset=['PCR'])

# Drop SUBJECTID (not a feature)
df = df.drop(columns=['SUBJECTID'])

# Separate features and target
X = df.drop(columns=['PCR'])
y = df['PCR'].astype(int)

# Fill any remaining NaNs in features with column mean
X = X.fillna(X.mean())

# Train/test split (80/20, stratified to handle class imbalance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Save for later steps
pickle.dump((X_train_scaled, X_test_scaled, y_train, y_test, X.columns.tolist(), scaler),
            open("preprocessed_data.pkl", "wb"))

print(" Preprocessing done!")
print(f"   Training samples: {X_train_scaled.shape[0]}")
print(f"   Test samples:     {X_test_scaled.shape[0]}")
print(f"   Features:         {X_train_scaled.shape[1]}")
print(f"   PCR=1 (train): {sum(y_train==1)}, PCR=0 (train): {sum(y_train==0)}")
print(f"   PCR=1 (test):  {sum(y_test==1)}, PCR=0 (test):  {sum(y_test==0)}")

**Step 2: Training ML Model**

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Class weight ratio for XGBoost (manual equivalent of 'balanced')
scale_pos_weight = 122 / 50  # NPCR count / PCR count

# Define models with class balancing
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    "SVM":                 SVC(kernel='rbf', probability=True, random_state=42, class_weight='balanced'),
    "Random Forest":       RandomForestClassifier(n_estimators=100, random_state=42, class_weight='balanced'),
    "Decision Tree":       DecisionTreeClassifier(random_state=42, class_weight='balanced'),
    "XGBoost":             XGBClassifier(eval_metric='logloss', random_state=42, scale_pos_weight=scale_pos_weight)
}

# Train and evaluate
results = {}
trained_models = {}

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    results[name] = {
        "Accuracy":  round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred, zero_division=0), 4),
        "Recall":    round(recall_score(y_test, y_pred, zero_division=0), 4),
        "F1-Score":  round(f1_score(y_test, y_pred, zero_division=0), 4),
        "y_pred":    y_pred
    }
    trained_models[name] = model
    print(f"{name} trained.")

# Print results table
print("\n--- Model Performance (Balanced) ---")
print(f"{'Model':<22} {'Accuracy':>10} {'Precision':>10} {'Recall':>10} {'F1-Score':>10}")
print("-" * 65)
for name, r in results.items():
    print(f"{name:<22} {r['Accuracy']:>10} {r['Precision']:>10} {r['Recall']:>10} {r['F1-Score']:>10}")

**Step 3: Confusion Matrices**

In [ ]:
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, r) in zip(axes, results.items()):
    cm = confusion_matrix(y_test, r['y_pred'])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['NPCR','PCR'], yticklabels=['NPCR','PCR'])
    ax.set_title(name, fontsize=10)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Actual')

plt.tight_layout()
plt.savefig("confusion_matrices.png", dpi=150)
plt.show()
print("Confusion matrices saved.")

Step 4: SHAP Explanantions

In [ ]:
!pip install shap -q

import shap
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

feature_names = X.columns.tolist()
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)

# ── Helper: safe extraction of 2D SHAP values for class 1 ──────
def get_shap_2d(shap_vals):
    if isinstance(shap_vals, list):
        sv = shap_vals[1]
    else:
        sv = shap_vals
    if sv.ndim == 3:
        sv = sv.sum(axis=2)
    return sv

# ── 1. Logistic Regression ─────────────────────────────────────
explainer_lr = shap.LinearExplainer(trained_models["Logistic Regression"],
                                     X_train_df, feature_perturbation="interventional")
sv_lr = np.array(explainer_lr.shap_values(X_test_df))
if sv_lr.ndim == 3:
    sv_lr = sv_lr[1]

plt.figure()
shap.summary_plot(sv_lr, X_test_df, plot_type="bar", max_display=15,
                  show=False, plot_size=(10, 5))
plt.title("Logistic Regression – SHAP Feature Importance (Top 15)")
plt.tight_layout()
plt.savefig("shap_lr_bar.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(sv_lr, X_test_df, max_display=15, show=False, plot_size=(10, 6))
plt.title("Logistic Regression – SHAP Summary Plot (Top 15)")
plt.tight_layout()
plt.savefig("shap_lr_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("Logistic Regression SHAP done.")

# ── 2. SVM (KernelExplainer) ───────────────────────────────────
svm_model = trained_models["SVM"]
def svm_predict(x):
    return svm_model.predict_proba(x)[:, 1]

background = shap.sample(X_train_df, 50, random_state=42)
sample_test = X_test_df.iloc[:20]

explainer_svm = shap.KernelExplainer(svm_predict, background)
shap_vals_svm = explainer_svm.shap_values(sample_test, nsamples=100)
sv_svm = np.array(shap_vals_svm)
if sv_svm.ndim == 3:
    sv_svm = sv_svm[0]

plt.figure()
shap.summary_plot(sv_svm, sample_test, plot_type="bar", max_display=15,
                  show=False, plot_size=(10, 5))
plt.title("SVM – SHAP Feature Importance (Top 15)")
plt.tight_layout()
plt.savefig("shap_svm_bar.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(sv_svm, sample_test, max_display=15, show=False, plot_size=(10, 6))
plt.title("SVM – SHAP Summary Plot (Top 15)")
plt.tight_layout()
plt.savefig("shap_svm_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("SVM SHAP done.")

# ── 3. Random Forest ───────────────────────────────────────────
explainer_rf = shap.TreeExplainer(trained_models["Random Forest"],
                                   feature_perturbation="tree_path_dependent")
sv_rf = get_shap_2d(explainer_rf.shap_values(X_test_df))

plt.figure()
shap.summary_plot(sv_rf, X_test_df, plot_type="bar", max_display=15,
                  show=False, plot_size=(10, 5))
plt.title("Random Forest – SHAP Feature Importance (Top 15)")
plt.tight_layout()
plt.savefig("shap_rf_bar.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(sv_rf, X_test_df, max_display=15, show=False, plot_size=(10, 6))
plt.title("Random Forest – SHAP Summary Plot (Top 15)")
plt.tight_layout()
plt.savefig("shap_rf_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("Random Forest SHAP done.")

# ── 4. Decision Tree ───────────────────────────────────────────
explainer_dt = shap.TreeExplainer(trained_models["Decision Tree"],
                                   feature_perturbation="tree_path_dependent")
sv_dt = get_shap_2d(explainer_dt.shap_values(X_test_df))

plt.figure()
shap.summary_plot(sv_dt, X_test_df, plot_type="bar", max_display=15,
                  show=False, plot_size=(10, 5))
plt.title("Decision Tree – SHAP Feature Importance (Top 15)")
plt.tight_layout()
plt.savefig("shap_dt_bar.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(sv_dt, X_test_df, max_display=15, show=False, plot_size=(10, 6))
plt.title("Decision Tree – SHAP Summary Plot (Top 15)")
plt.tight_layout()
plt.savefig("shap_dt_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("Decision Tree SHAP done.")

# ── 5. XGBoost ─────────────────────────────────────────────────
explainer_xgb = shap.TreeExplainer(trained_models["XGBoost"])
sv_xgb = get_shap_2d(explainer_xgb.shap_values(X_test_df))

plt.figure()
shap.summary_plot(sv_xgb, X_test_df, plot_type="bar", max_display=15,
                  show=False, plot_size=(10, 5))
plt.title("XGBoost – SHAP Feature Importance (Top 15)")
plt.tight_layout()
plt.savefig("shap_xgb_bar.png", dpi=150, bbox_inches='tight')
plt.show()

plt.figure()
shap.summary_plot(sv_xgb, X_test_df, max_display=15, show=False, plot_size=(10, 6))
plt.title("XGBoost – SHAP Summary Plot (Top 15)")
plt.tight_layout()
plt.savefig("shap_xgb_summary.png", dpi=150, bbox_inches='tight')
plt.show()
print("XGBoost SHAP done.")

**Step 5: LIME Explanations**

In [ ]:
!pip install lime -q

import lime
import lime.lime_tabular
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

feature_names = X.columns.tolist()
X_test_df = pd.DataFrame(X_test_scaled, columns=feature_names)
X_train_df = pd.DataFrame(X_train_scaled, columns=feature_names)

# ── Pick one PCR=1 and one PCR=0 sample from test set ──────────
y_test_arr = np.array(y_test)
idx_pcr1 = np.where(y_test_arr == 1)[0][0]   # first PCR=1 patient
idx_pcr0 = np.where(y_test_arr == 0)[0][0]   # first PCR=0 patient
sample_indices = {"PCR=1 (Responder)": idx_pcr1, "PCR=0 (Non-Responder)": idx_pcr0}

# ── LIME explainer (shared across all models) ──────────────────
explainer = lime.lime_tabular.LimeTabularExplainer(
    training_data=X_train_scaled,
    feature_names=feature_names,
    class_names=["NPCR", "PCR"],
    mode="classification",
    random_state=42
)

model_list = {
    "Logistic_Regression": trained_models["Logistic Regression"],
    "SVM":                 trained_models["SVM"],
    "Random_Forest":       trained_models["Random Forest"],
    "Decision_Tree":       trained_models["Decision Tree"],
    "XGBoost":             trained_models["XGBoost"]
}

# ── Generate and save LIME plots ───────────────────────────────
for model_name, model in model_list.items():
    for label, idx in sample_indices.items():
        sample = X_test_scaled[idx]

        exp = explainer.explain_instance(
            data_row=sample,
            predict_fn=model.predict_proba,
            num_features=10,
            top_labels=1
        )

        fig = exp.as_pyplot_figure(label=exp.available_labels()[0])
        plt.title(f"{model_name.replace('_',' ')} – LIME\n{label}", fontsize=11)
        plt.tight_layout()

        fname = f"lime_{model_name}_{label.split('=')[1][0]}.png"
        plt.savefig(fname, dpi=150, bbox_inches='tight')
        plt.show()
        plt.close()
        print(f"Saved: {fname}")